# P-2026-11-13-1: Can HS6 ECI be rescued from localization?

**Question.** At HS6, the standard ECI collapses: the nightly Q04 run (open PR #4) found R² ≈ 0.002 of log GDP per capita on HS6 ECI in 2019, because the eigenvector concentrates on Gabon and Suriname. Does a **minimum-ubiquity product filter** or a **regularized eigenvector** remove that localization, and does HS6 ECI then explain income better than HS4?

- **Source:** proposal 7 in `proposals/2026-11-12-all-questions.md` (PR #4 branch). `QUEUE.md` Q01–Q16 are all taken: Q01–Q02 on `main`, Q03–Q16 in open PR #4. So, following PROTOCOL §1.4, tonight's question comes from the proposals.
- **Date:** 2026-11-13 (nightly run)
- **Data:** `data/raw/<year>_mega.csv` (HS6, 2019–2023), aggregated to HS4 for the baseline. `maitreemody/outputs/worldbank_gdp_per_capita_2019_2023.csv` (GDP per capita, PPP). Same sample rules as Q04: reporters with at least $1B in exports, HS chapters 98/99 dropped, regional aggregates dropped, and only countries with GDP data. 2019 is primary. 2020–2023 are checks, and 2023 is suspect (D1).
- **Run time:** the notebook executes in about 10 s (last cell prints the exact runtime).

## TL;DR

- **The HS6 collapse comes from 53 single-exporter products.** In 2019, 53 of the 5,490 HS6 products have RCA ≥ 1 in only one country, and Gabon holds 21 of them. The plain HS6 eigenvector is almost a Gabon indicator: max |ECI| is 10.35 (GAB), and IPR×n is 103.0, against 2.48 at HS4. Dropping only those 53 products (pre-stated u ≥ 2) raises R² from **0.002 to 0.569**. With u ≥ 3, R² is **0.628**, against **0.583 for HS4**.
- **Even after the fix, HS6 does not reliably beat HS4.** With the u ≥ 3 filter, HS6 minus HS4 R² is +0.045 (95% bootstrap interval −0.004 to 0.099) in 2019, +0.034 (−0.016 to 0.082) in 2021, and **−0.092 (−0.209 to 0.023) in 2022**. HS6 u ≥ 2 falls to 0.480 and 0.450 in 2022 and 2023, while HS4 stays at 0.583.
- **The pre-stated regularization (α = 1) makes things worse.** R² is 0.398 in 2019, below HS4 in every year, and its Spearman correlation with HS4 ECI is only 0.630 (Vietnam drops to rank 129 of 132). A post-hoc weak setting (α = 0.1) reaches 0.597 in 2019 but does not beat HS4 either.
- **Verdict for the lab:** HS4 stays the right level. If anyone uses HS6, they should always drop products with ubiquity < 2 (≥ 3 or 5 is safer). At that point HS6 ranks agree with HS4 at Spearman 0.964. Most Spring 2026 report countries move 0–3 ranks, but Canada (44 → 23) and Vietnam (65 → 83) move a lot.

## Why this matters for the lab

Q04 showed that the lab's 18-group ECI is far from a usable income signal, while HS4 recovers it. The obvious next question is whether we should go **finer** than HS4. If HS6 were better once fixed, the migration target for the reports (PR #4 proposal 1) would change. This run tests the **implementation** link, "is our ECI measuring what it should?", and decides the product level for any port of the reports. It also produces a guard rule (minimum ubiquity) that any fine-grained ECI in the lab should apply.

## Method

**Pre-stated metric:** R² of log GDP per capita (PPP) on ECI, cross-section, 2019, using the same countries at every level (n = 132). Secondary metrics: Spearman rank correlation with income, year-to-year rank stability, and localization diagnostics (max |ECI| and IPR×n, where IPR is the inverse participation ratio of the eigenvector. IPR×n ≈ 1–3 means the eigenvector is spread out, and IPR×n ≈ n means it sits on one country).

**Baseline:** standard ECI (binary RCA ≥ 1, sorted eigenvalues, sign oriented to diversity) at **HS4** and at plain **HS6**.

**Variants (all reported):**
1. *Minimum-ubiquity filter:* drop products with k_p < u, for u ∈ {1, 2, 3, 5, 10, 20}. **Pre-stated choice: u = 2** (drop single-exporter products, the smallest principled fix). Applied to HS4 as well for fairness.
2. *Regularized eigenvector* (Qin & Rohe 2013 style): add τ = α × mean degree to both country and product degrees before the normalized SVD, for α ∈ {0, 0.1, 0.25, 0.5, 1, 2}. **Pre-stated choice: α = 1** (τ = mean degree, the usual default).
3. A combination (u ≥ 2 plus α = 1), and two **post-hoc** settings picked after seeing the 2019 sweeps (u ≥ 5, α = 0.1). These are labelled `[post-hoc]` and should not be read as confirmatory.

**Checks:** all five years, a bootstrap of the R² gap against HS4 (countries resampled 2,000 times for the regression, with ECI held fixed), year-to-year Spearman stability, agreement with HS4, and the ranks of the Spring 2026 report countries.

In [1]:
import sys, time, warnings
T0 = time.time()
warnings.filterwarnings('ignore')
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import eci
pd.set_option('display.width', 160); pd.set_option('display.precision', 4)
REPO = eci.REPO
YEARS = [2019, 2020, 2021, 2022, 2023]
ISO_FIX = {381: 'ITA'}   # lib gap (data-requests/2026-11-12): Comtrade 381 = Italy

def load_hs6(year, min_exports=1e9):
    """HS6 exports (ISO3 x HS6) from data/raw, same rules as eci.load_hs4 (drop ch. 98/99,
    regional aggregates, reporters < $1B), keyed by ISO3."""
    raw = pd.read_csv(REPO / 'data/raw' / f'{year}_mega.csv', index_col=0)
    cmd = raw.pop('commodity_code') if 'commodity_code' in raw.columns else pd.Series(raw.index, index=raw.index)
    cmd = cmd.astype(str).str.zfill(6)
    raw = raw.apply(pd.to_numeric, errors='coerce').fillna(0)
    keep = ~cmd.str[:2].isin(['98', '99'])
    X = raw[keep].groupby(cmd[keep].values).sum().T
    X.index = X.index.astype(int)
    X = X.drop(index=[c for c in X.index if c in eci.M49_AGGREGATES])
    X = X.loc[X.sum(axis=1) >= min_exports]
    X.index = [ISO_FIX.get(c) or eci.m49_to_iso3(c) for c in X.index]
    return X.loc[X.index.notna()]

gdp = pd.read_csv(REPO / 'maitreemody/outputs/worldbank_gdp_per_capita_2019_2023.csv')
def log_gdppc(year):
    return np.log(gdp[gdp.year == year].set_index('iso3').gdppc_ppp.dropna())

def eci_variant(M, min_ubiq=1, tau_alpha=0.0):
    """Oriented ECI from binary M. min_ubiq: drop products with ubiquity < u (then countries
    with no products). tau_alpha: regularised eigenvector, degrees kc+tau_c, kp+tau_p with
    tau = alpha * mean degree (Qin & Rohe 2013); alpha=0 is the standard ECI."""
    kp = M.sum(axis=0); M = M.loc[:, kp >= min_ubiq]; M = M.loc[M.sum(axis=1) > 0]
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    tc, tp = tau_alpha * kc.mean(), tau_alpha * kp.mean()
    B = Mv / np.sqrt((kc + tc)[:, None] * (kp + tp)[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc + tc)
    if np.corrcoef(k, kc)[0, 1] < 0: k = -k
    v = U[:, 1]
    ipr = float((v**4).sum() / (v**2).sum()**2)   # inverse participation ratio: 1/n = spread, 1 = one country
    lam = S**2
    e = pd.Series((k - k.mean()) / k.std(), index=M.index)
    return e, {'n_products': M.shape[1], 'n_countries': M.shape[0], 'ipr': ipr, 'ipr_x_n': ipr * M.shape[0],
               'eigengap_23': float(lam[1] - lam[2]), 'max_abs_eci': float(e.abs().max()), 'max_abs_country': e.abs().idxmax()}

def fit(e, year):
    d = pd.concat([e.rename('eci'), log_gdppc(year).rename('y')], axis=1, join='inner').dropna()
    r = d.corr()
    return {'r2': float(r.iloc[0, 1]**2), 'spearman': float(d.corr('spearman').iloc[0, 1]), 'n': len(d)}

DATA = {}
for y in YEARS:
    X6 = load_hs6(y)
    X6 = X6.loc[X6.index.intersection(log_gdppc(y).index)]   # same sample at every level (as Q04)
    X4 = X6.T.groupby(X6.columns.str[:4]).sum().T
    DATA[y] = {'HS6': eci.mcp(X6), 'HS4': eci.mcp(X4)}       # RCA >= 1, standard (Atlas) rule
    print(y, 'countries', X6.shape[0], '| HS6 products', DATA[y]['HS6'].shape[1], '| HS4 products', DATA[y]['HS4'].shape[1])


2019 countries 132 | HS6 products 5490 | HS4 products 1224
2020 countries 129 | HS6 products 5467 | HS4 products 1223


2021 countries 135 | HS6 products 5465 | HS4 products 1223
2022 countries 118 | HS6 products 5790 | HS4 products 1231


2023 countries 130 | HS6 products 5812 | HS4 products 1230


## Results
### A. Baseline: reproduce the HS6 collapse (2019)

In [2]:
base = {}
for lvl in ['HS6', 'HS4']:
    e, d = eci_variant(DATA[2019][lvl]); base[lvl] = e
    print(lvl, {**fit(e, 2019), **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()}})
print('\nTop 5 |ECI| at HS6 (2019):'); print(base['HS6'].reindex(base['HS6'].abs().sort_values(ascending=False).index).head(5).round(2).to_string())


HS6 {'r2': 0.0017712271662287597, 'spearman': 0.5597825236234236, 'n': 132, 'n_products': 5490, 'n_countries': 132, 'ipr': 0.7803, 'ipr_x_n': 103.0048, 'eigengap_23': 0.0781, 'max_abs_eci': 10.3547, 'max_abs_country': 'GAB'}
HS4 {'r2': 0.5826283794181267, 'spearman': 0.7836663135980131, 'n': 132, 'n_products': 1224, 'n_countries': 132, 'ipr': 0.0188, 'ipr_x_n': 2.4812, 'eigengap_23': 0.0774, 'max_abs_eci': 2.387, 'max_abs_country': 'JPN'}

Top 5 |ECI| at HS6 (2019):
GAB   -10.35
SUR    -4.53
QAT    -0.60
COG    -0.47
COD    -0.46


Takeaway: the plain HS6 ECI has essentially no relationship with income because the second eigenvector piles onto one or two countries (see max |ECI| and IPR).

In [3]:
M6 = DATA[2019]['HS6']; kp = M6.sum(axis=0)
print('HS6 products by ubiquity (2019):', {f'k_p={u}': int((kp == u).sum()) for u in [1, 2, 3, 4]}, '| total', len(kp))
single = M6.loc[:, kp == 1]
own = single.sum(axis=0).pipe(lambda s: single.idxmax(axis=0)).value_counts()
print('\nCountries owning the most singleton (ubiquity-1) HS6 products:'); print(own.head(8).to_string())
for c in ['GAB', 'SUR']:
    if c in M6.index:
        print(f'{c}: diversity {int(M6.loc[c].sum())}, of which singletons {int(single.loc[c].sum())}')


HS6 products by ubiquity (2019): {'k_p=1': 53, 'k_p=2': 69, 'k_p=3': 116, 'k_p=4': 146} | total 5490

Countries owning the most singleton (ubiquity-1) HS6 products:
GAB    21
SUR     5
MEX     3
CHN     3
UKR     3
EGY     2
USA     2
SAU     2
GAB: diversity 95, of which singletons 21
SUR: diversity 48, of which singletons 5


Takeaway: the extreme countries are the ones that hold products no other country has RCA in. A product with ubiquity 1 links to exactly one country, which lets the eigenvector isolate it.

### B. Variant 1: minimum-ubiquity filter (drop products with k_p < u). Pre-stated choice: u = 2

In [4]:
rows = []
for lvl in ['HS6', 'HS4']:
    for u in [1, 2, 3, 5, 10, 20]:
        e, d = eci_variant(DATA[2019][lvl], min_ubiq=u)
        rows.append({'level': lvl, 'min_ubiq': u, **fit(e, 2019), **d})
ub = pd.DataFrame(rows)
print(ub[['level', 'min_ubiq', 'n_products', 'n_countries', 'r2', 'spearman', 'ipr_x_n', 'eigengap_23', 'max_abs_eci', 'max_abs_country']].round(4).to_string(index=False))
ub.to_csv('figures/ubiquity_sweep_2019.csv', index=False)


level  min_ubiq  n_products  n_countries     r2  spearman  ipr_x_n  eigengap_23  max_abs_eci max_abs_country
  HS6         1        5490          132 0.0018    0.5598 103.0048       0.0781      10.3547             GAB
  HS6         2        5437          132 0.5689    0.8001   2.6329       0.0616       2.6560             SUR
  HS6         3        5368          132 0.6281    0.8136   2.6793       0.0678       2.4014             JPN
  HS6         5        5106          132 0.6329    0.8193   2.5660       0.0676       2.3234             JPN
  HS6        10        3858          132 0.6306    0.8159   2.3312       0.0647       2.1455             JPN
  HS6        20        1209          132 0.4260    0.7298   3.3494       0.0312       2.9224             KHM
  HS4         1        1224          132 0.5826    0.7837   2.4812       0.0774       2.3870             JPN
  HS4         2        1223          132 0.5828    0.7840   2.4818       0.0775       2.3870             JPN
  HS4         3    

Takeaway: removing singleton products alone is enough to undo the localization at HS6; see the table for whether HS6 then matches or beats HS4.

### C. Variant 2: regularized eigenvector (no product filter). Pre-stated choice: alpha = 1 (tau = mean degree)

In [5]:
rows = []
for lvl in ['HS6', 'HS4']:
    for a in [0, 0.1, 0.25, 0.5, 1, 2]:
        e, d = eci_variant(DATA[2019][lvl], tau_alpha=a)
        rows.append({'level': lvl, 'tau_alpha': a, **fit(e, 2019), **d})
rg = pd.DataFrame(rows)
print(rg[['level', 'tau_alpha', 'r2', 'spearman', 'ipr_x_n', 'eigengap_23', 'max_abs_eci', 'max_abs_country']].round(4).to_string(index=False))
rg.to_csv('figures/regularization_sweep_2019.csv', index=False)


level  tau_alpha     r2  spearman  ipr_x_n  eigengap_23  max_abs_eci max_abs_country
  HS6       0.00 0.0018    0.5598 103.0048       0.0781      10.3547             GAB
  HS6       0.10 0.5970    0.8264   2.9302       0.0602       2.6007             JPN
  HS6       0.25 0.5221    0.7851   3.0732       0.0498       2.6307             JPN
  HS6       0.50 0.4582    0.7507   3.2130       0.0367       2.6562             JPN
  HS6       1.00 0.3985    0.7168   3.3978       0.0221       2.7597             DEU
  HS6       2.00 0.3484    0.6819   3.6393       0.0107       3.0869             DEU
  HS4       0.00 0.5826    0.7837   2.4812       0.0774       2.3870             JPN
  HS4       0.10 0.6088    0.8181   2.6197       0.0633       2.6130             JPN
  HS4       0.25 0.5786    0.8062   2.7109       0.0492       2.7176             JPN
  HS4       0.50 0.5385    0.7926   2.8063       0.0351       2.7943             JPN
  HS4       1.00 0.4915    0.7676   2.9354       0.0208       2.8

Takeaway: regularization is the alternative fix that needs no product to be thrown away; compare its R² with variant 1 at the pre-stated settings.

### D. Do the pre-stated fixes hold in every year? (2023 suspect, D1)

In [6]:
SPECS = {'HS4 (baseline)': ('HS4', 1, 0), 'HS4 u>=2': ('HS4', 2, 0), 'HS6 plain': ('HS6', 1, 0),
         'HS6 u>=2': ('HS6', 2, 0), 'HS6 u>=3': ('HS6', 3, 0), 'HS6 reg a=1': ('HS6', 1, 1.0), 'HS6 u>=2 + reg a=1': ('HS6', 2, 1.0),
         # post-hoc (chosen after seeing 2019 sweeps; labelled as such):
         'HS6 u>=5 [post-hoc]': ('HS6', 5, 0), 'HS6 reg a=0.1 [post-hoc]': ('HS6', 1, 0.1)}
ECI = {}; rows = []
for y in YEARS:
    for name, (lvl, u, a) in SPECS.items():
        e, d = eci_variant(DATA[y][lvl], min_ubiq=u, tau_alpha=a); ECI[(name, y)] = e
        rows.append({'year': y, 'variant': name, **fit(e, y), 'max_abs_country': d['max_abs_country'], 'max_abs_eci': d['max_abs_eci'], 'ipr_x_n': d['ipr_x_n']})
yr = pd.DataFrame(rows)
print(yr.pivot(index='variant', columns='year', values='r2').loc[list(SPECS)].round(3).to_string())
print('\nn per year:', yr.groupby('year').n.first().to_dict())
print('\nLargest |ECI| by variant/year:'); print(yr.pivot(index='variant', columns='year', values='max_abs_eci').loc[list(SPECS)].round(2).to_string())
print('\nMost extreme country by variant/year:'); print(yr.pivot(index='variant', columns='year', values='max_abs_country').loc[list(SPECS)].to_string())
yr.to_csv('figures/variants_by_year.csv', index=False)


year                       2019   2020   2021   2022   2023
variant                                                    
HS4 (baseline)            0.583  0.625  0.556  0.583  0.583
HS4 u>=2                  0.583  0.627  0.556  0.584  0.583
HS6 plain                 0.002  0.250  0.278  0.362  0.369
HS6 u>=2                  0.569  0.646  0.584  0.480  0.450
HS6 u>=3                  0.628  0.647  0.590  0.491  0.479
HS6 reg a=1               0.398  0.410  0.397  0.438  0.414
HS6 u>=2 + reg a=1        0.399  0.410  0.397  0.438  0.415
HS6 u>=5 [post-hoc]       0.633  0.654  0.598  0.503  0.535
HS6 reg a=0.1 [post-hoc]  0.597  0.602  0.583  0.559  0.558

n per year: {2019: 132, 2020: 129, 2021: 135, 2022: 118, 2023: 130}

Largest |ECI| by variant/year:
year                       2019  2020  2021  2022  2023
variant                                                
HS4 (baseline)             2.39  2.41  2.41  2.42  2.70
HS4 u>=2                   2.39  2.41  2.41  2.42  2.70
HS6 plain      

Takeaway: this is the out-of-sample check for the 2019-chosen settings.

In [7]:
# Why does 2022 still break at u>=2/3? Look at the low-ubiquity products of the extreme country.
for y in [2022, 2023]:
    M = DATA[y]['HS6']; kp = M.sum(axis=0)
    for c in ['AGO', 'SUR', 'GAB']:
        if c in M.index:
            row = M.loc[c]; prods = row[row > 0].index
            print(y, c, 'diversity', int(row.sum()), '| products with ubiquity 1/2/3/4:', [int((kp[prods] == u).sum()) for u in [1, 2, 3, 4]])


2022 AGO diversity 67 | products with ubiquity 1/2/3/4: [1, 0, 2, 1]
2022 SUR diversity 148 | products with ubiquity 1/2/3/4: [20, 5, 2, 2]
2023 AGO diversity 101 | products with ubiquity 1/2/3/4: [2, 1, 2, 3]
2023 SUR diversity 81 | products with ubiquity 1/2/3/4: [5, 3, 2, 2]
2023 GAB diversity 74 | products with ubiquity 1/2/3/4: [0, 0, 0, 0]


In [8]:
# Uncertainty on the R^2 gap vs HS4 (countries resampled for the regression only; ECI held fixed)
rng = np.random.default_rng(0); B = 2000; out = []
for y in [2019, 2021, 2022]:
    yv = log_gdppc(y)
    for name in ['HS6 u>=2', 'HS6 u>=3', 'HS6 reg a=1', 'HS6 u>=5 [post-hoc]', 'HS6 reg a=0.1 [post-hoc]']:
        d = pd.concat([ECI[(name, y)].rename('a'), ECI[('HS4 (baseline)', y)].rename('b'), yv.rename('y')], axis=1, join='inner').dropna().to_numpy()
        diffs = []
        for _ in range(B):
            s = d[rng.integers(0, len(d), len(d))]
            diffs.append(np.corrcoef(s[:, 0], s[:, 2])[0, 1]**2 - np.corrcoef(s[:, 1], s[:, 2])[0, 1]**2)
        point = np.corrcoef(d[:, 0], d[:, 2])[0, 1]**2 - np.corrcoef(d[:, 1], d[:, 2])[0, 1]**2
        out.append({'year': y, 'variant': name, 'r2_minus_hs4': point, 'ci_lo': np.percentile(diffs, 2.5), 'ci_hi': np.percentile(diffs, 97.5), 'share_boot_gt0': np.mean(np.array(diffs) > 0)})
boot = pd.DataFrame(out); print(boot.round(4).to_string(index=False))


 year                  variant  r2_minus_hs4   ci_lo   ci_hi  share_boot_gt0
 2019                 HS6 u>=2       -0.0137 -0.0898  0.0625          0.3980
 2019                 HS6 u>=3        0.0454 -0.0036  0.0989          0.9610
 2019              HS6 reg a=1       -0.1841 -0.3044 -0.0623          0.0035
 2019      HS6 u>=5 [post-hoc]        0.0503  0.0018  0.1006          0.9790
 2019 HS6 reg a=0.1 [post-hoc]        0.0143 -0.0730  0.1005          0.6145
 2021                 HS6 u>=2        0.0271 -0.0221  0.0725          0.8515
 2021                 HS6 u>=3        0.0336 -0.0156  0.0824          0.9170
 2021              HS6 reg a=1       -0.1599 -0.2942 -0.0146          0.0165
 2021      HS6 u>=5 [post-hoc]        0.0420 -0.0013  0.0867          0.9705
 2021 HS6 reg a=0.1 [post-hoc]        0.0262 -0.0661  0.1228          0.6965
 2022                 HS6 u>=2       -0.1032 -0.2223  0.0169          0.0505
 2022                 HS6 u>=3       -0.0915 -0.2085  0.0228          0.0655

Takeaway: whether HS6 beats HS4 by more than sampling noise in the regression.

### E. Year-to-year rank stability

In [9]:
rows = []
for name in SPECS:
    for y0, y1 in zip(YEARS[:-1], YEARS[1:]):
        a, b = ECI[(name, y0)], ECI[(name, y1)]; common = a.index.intersection(b.index)
        rows.append({'variant': name, 'pair': f'{y0}-{y1}', 'spearman': a[common].corr(b[common], method='spearman'), 'n': len(common)})
st = pd.DataFrame(rows)
print(st.pivot(index='variant', columns='pair', values='spearman').loc[list(SPECS)].round(3).to_string())


pair                      2019-2020  2020-2021  2021-2022  2022-2023
variant                                                             
HS4 (baseline)                0.978      0.960      0.981      0.981
HS4 u>=2                      0.977      0.960      0.981      0.981
HS6 plain                     0.790      0.965      0.943      0.957
HS6 u>=2                      0.970      0.972      0.944      0.955
HS6 u>=3                      0.977      0.974      0.944      0.957
HS6 reg a=1                   0.996      0.991      0.977      0.968
HS6 u>=2 + reg a=1            0.996      0.991      0.977      0.969
HS6 u>=5 [post-hoc]           0.978      0.974      0.941      0.960
HS6 reg a=0.1 [post-hoc]      0.991      0.988      0.930      0.955


Takeaway: a fix that only helps the income fit but makes ranks jumpy would not be usable in reports.

### F. Agreement with HS4 and the Spring 2026 report countries

In [10]:
y = 2019; h4 = ECI[('HS4 (baseline)', y)]
for name in ['HS6 u>=2', 'HS6 u>=3', 'HS6 reg a=1', 'HS6 u>=5 [post-hoc]', 'HS6 reg a=0.1 [post-hoc]']:
    e = ECI[(name, y)]; c = e.index.intersection(h4.index)
    print(f'{name}: Spearman with HS4 ECI = {e[c].corr(h4[c], method="spearman"):.3f} (n={len(c)})')
REP = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
tab = pd.DataFrame({name: ECI[(name, y)].rank(ascending=False) for name in ['HS4 (baseline)', 'HS6 plain', 'HS6 u>=2', 'HS6 u>=3', 'HS6 u>=5 [post-hoc]', 'HS6 reg a=1']}).reindex(REP)
print('\nECI rank (1 = most complex) of Spring 2026 report countries, 2019, n =', len(h4)); print(tab.to_string())


HS6 u>=2: Spearman with HS4 ECI = 0.964 (n=132)
HS6 u>=3: Spearman with HS4 ECI = 0.962 (n=132)
HS6 reg a=1: Spearman with HS4 ECI = 0.630 (n=132)
HS6 u>=5 [post-hoc]: Spearman with HS4 ECI = 0.960 (n=132)
HS6 reg a=0.1 [post-hoc]: Spearman with HS4 ECI = 0.852 (n=132)

ECI rank (1 = most complex) of Spring 2026 report countries, 2019, n = 132
     HS4 (baseline)  HS6 plain  HS6 u>=2  HS6 u>=3  HS6 u>=5 [post-hoc]  HS6 reg a=1
CAN            44.0       51.0      23.0      23.0                 22.0         17.0
DEU             5.0        4.0       3.0       3.0                  3.0          1.0
POL            27.0       19.0      26.0      26.0                 26.0         23.0
SGP             4.0       38.0       5.0       5.0                  5.0         11.0
ARE            55.0       75.0      55.0      56.0                 57.0         66.0
GBR            11.0       20.0      14.0      14.0                 14.0         10.0
VNM            65.0       49.0      83.0      84.0         

Takeaway: how much the report countries would move if the lab adopted HS6 with a ubiquity filter instead of HS4.

In [11]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for lvl, mk in [('HS6', 'o'), ('HS4', 's')]:
    d = ub[ub.level == lvl]; ax[0].plot(d.min_ubiq, d.r2, marker=mk, label=lvl)
    d = rg[rg.level == lvl]; ax[1].plot(d.tau_alpha, d.r2, marker=mk, label=lvl)
ax[0].set_xscale('log'); ax[0].set_xticks([1, 2, 3, 5, 10, 20]); ax[0].set_xticklabels([1, 2, 3, 5, 10, 20])
ax[0].set_xlabel('minimum product ubiquity u'); ax[1].set_xlabel('regularization alpha (tau = alpha x mean degree)')
for a in ax: a.set_ylabel('R² of log GDPpc (PPP) on ECI, 2019'); a.legend(); a.grid(alpha=.3)
ax[0].set_title('Variant 1: ubiquity filter'); ax[1].set_title('Variant 2: regularized eigenvector')
fig.tight_layout(); fig.savefig('figures/r2_sweeps_2019.png', dpi=110); plt.close(fig)
print('saved figures/r2_sweeps_2019.png'); print(f'runtime {time.time() - T0:.1f} s')


saved figures/r2_sweeps_2019.png
runtime 9.3 s


## Caveats

- **The income fit is a cross-section.** R² against log GDP per capita checks that ECI measures something income-related. It does not test growth prediction or causality.
- **Settings were chosen on 2019.** u = 2 and α = 1 were pre-stated. u ≥ 3, u ≥ 5 and α = 0.1 look better *because* they were picked after seeing 2019. 2020–2022 are the honest check, and there HS6 does not hold its lead over HS4.
- **The bootstrap holds ECI fixed** and resamples only the regression. It ignores the uncertainty in the eigenvector itself (Q09 in PR #4 covers that), so the intervals are too narrow, if anything.
- **The sample changes by year** (132 / 129 / 135 / 118 / 130 countries). 2022 has the smallest sample, and that is the year HS6 underperforms.
- **The data is exporter-reported**, with each reporter's own HS revision, nominal values, and no mirror data. 2023 totals are inflated (D1).
- GDP per capita comes from Maitree's World Bank extract (PPP). Italy (Comtrade 381) is mapped to ITA inside this notebook (the library gap from `data-requests/2026-11-12-eu28-and-italy-codes.md`). `lib/eci.py` was not modified.

## Helpful to know

- **Always apply a ubiquity floor when working above HS4.** At HS4 only one product in 2019 has ubiquity 1 (1,224 → 1,223), so the filter changes almost nothing there (R² 0.583 → 0.583). At HS6 it decides between garbage and a working index.
- **IPR×n is a cheap localization alarm.** Plain HS6 gives 103.0 in 2019, while every working variant gives about 2.3–3.6. Any future ECI variant should print it next to max |ECI|.
- **The 2022 HS6 dip is not the same failure.** In 2022 the extreme country under u ≥ 2 is Angola, with max |ECI| of only 2.54. Angola has just 1 / 0 / 2 / 1 products of ubiquity 1 / 2 / 3 / 4. HS6 product counts jump from 5,465 (2021) to 5,790 (2022) and 5,812 (2023), against 1,223 → 1,231 at HS4. That points to the **HS 2022 revision**: reporters on different HS editions produce different HS6 codes for the same goods. This is a hypothesis. The raw files don't record the classification, so it can't be tested from the repo.
- **Strong regularization rewards size, not complexity.** At α = 1, Germany becomes the top country every year from 2019 to 2022, and Vietnam falls to 129th. Large τ pulls the eigenvector toward the degree (diversity) direction.
- The regularized variants have the most stable ranks over time (0.968–0.996), but that comes from shrinkage toward diversity, not from measuring better.

## New research avenues

Each is one night, using repo data (also in `proposals/2026-11-13.md`):

1. **HS-revision check on the 2022 raw data.** For each 2022 HS6 code, count the reporters that use it, and flag codes that exist only in HS2022 or only in HS2017 (the code lists are public, but they would need to be added as a small reference file; see the data request). Then recompute HS6 u ≥ 3 R² on HS6 codes common to 2021 and 2022. This tests the hypothesis in "Helpful to know".
2. **Ubiquity floor at HS4 for the reports.** For the 7 Spring 2026 countries plus peers, check whether u ≥ 2 or u ≥ 3 at HS4 changes any rank by more than 2. The expected answer is no, which would let PR #4 proposal 1 adopt the floor as a free safety guard.
3. **Canada and Vietnam rank swings.** Canada goes 44 → 23 and Vietnam 65 → 83 between HS4 and filtered HS6. Find which products drive the swing: the HS6 codes with the largest contribution, M_cp × PCI, that differ from their HS4 parent. This says whether HS4 hides real specialisation in these two countries.
4. **Value floor instead of ubiquity floor.** Drop products with world exports below $X (e.g. $10M or $50M) instead of using ubiquity. This is the Atlas-style filter, and it may be less sensitive to the reporting artefacts behind the 2022 dip.

## Data issues / data needed

- **HS revision mix in the raw pulls (suspected, not verified).** HS6 product counts jump in 2022 (5,465 → 5,790), and HS6 ECI's income fit drops that year while HS4's does not. The raw `<year>_mega.csv` files have no classification column, so the repo can't show which reporters used HS2017 and which used HS2022. Request: `willbetts/nightly/data-requests/2026-11-13-hs-revision-mix.md`.
- **Italy ISO3 gap** in `lib/eci.py` (Comtrade 381). It is already reported in `data-requests/2026-11-12-eu28-and-italy-codes.md`, and is worked around locally here.
